# Week 1 Lab: Python Concurrency and Parallel Processing

**ISM 6562 — Big Data for Business Applications**

Dr. Tim Smith

# Introduction

## The Problem: Your Python Code Only Uses One Core

Up to this point in your studies, the Python code you've written has been **sequential** — a single process working through a problem one step at a time. Even if your computer has 8, 16, or more CPU cores, a standard Python script uses just **one** of them. The rest sit idle.

Open your system's activity monitor while running a Python script and you'll see it: one core pegged at 100%, the others doing nothing. This means you're leaving the vast majority of your computer's processing power on the table.

For small datasets and simple tasks, this doesn't matter. But for Big Data — terabytes of records, millions of computations — sequential processing simply cannot keep up. The entire Big Data ecosystem is built on the idea of **distributing work across many processors** to solve this problem.

### You May Have Already Used Parallel Processing Without Knowing It

Some Python libraries handle parallelism for you behind the scenes. For example, scikit-learn's `RandomForestClassifier` has an `n_jobs` parameter that controls how many CPU cores to use:

In [1]:
from sklearn.ensemble import RandomForestClassifier

# Uses only 1 core (default) -- sequential
model_slow = RandomForestClassifier(n_estimators=1000, n_jobs=1)

# Uses all available cores -- parallel
model_fast = RandomForestClassifier(n_estimators=1000, n_jobs=-1)

Setting `n_jobs=-1` tells scikit-learn to use all available CPU cores, and you'll see a significant speedup when training on large datasets. But this only works because the library authors wrote the parallelism code for you. In this lab, you'll learn how to do it yourself.

## What You'll Learn

In this lab, you will:

1. Understand Python's Global Interpreter Lock (GIL) and why it matters
2. Use `multiprocessing` to run tasks in parallel across multiple cores
3. Measure the speedup from parallel vs sequential execution
4. Connect local parallelism to distributed computing concepts

## ⚙️ Environment Setup

This lab uses the `multiprocess` library (a fork of Python's built-in `multiprocessing` that works better in Jupyter notebooks).

**Run the cell below to install it:**

In [2]:
!pip install -q multiprocess

# Python's GIL: Understanding the Limitation

## What is the GIL?

Python (specifically CPython, the standard implementation) has a **Global Interpreter Lock (GIL)** — a mutex that allows only one thread to execute Python bytecode at a time.

The GIL was implemented to:

- Ensure thread safety for Python's memory management
- Simplify the CPython interpreter implementation
- Prevent race conditions in the interpreter itself

This means even on a multi-core system, **Python threads cannot utilize multiple CPU cores for CPU-bound tasks**. Only one thread can execute Python code at any given moment.

## The Solution: Multiprocessing

The `multiprocessing` module bypasses the GIL by creating **separate Python processes**, each with its own interpreter and memory space. Since each process has its own GIL, they can truly run in parallel on different CPU cores.

This is the approach we'll use in this lab — and it's the same fundamental pattern used by distributed Big Data frameworks.

# Hands-On: Sequential vs Parallel Processing

## Step 1: Import Libraries and Check Your System

In [3]:
import multiprocess
from multiprocess import Pool
import time

# Check your system's capabilities
number_of_cores = multiprocess.cpu_count()

print(f"Your System Information:")
print(f"  CPU Cores Available: {number_of_cores}")
print(f"  Recommended for Processing: {number_of_cores - 1} cores (leaving 1 for OS)")

Your System Information:
  CPU Cores Available: 16
  Recommended for Processing: 15 cores (leaving 1 for OS)


## Step 2: Configure Processing Parameters

In [4]:
# How many cores to use (leave 1 for the OS)
number_of_cores_to_use = max(1, number_of_cores - 1)

# Workload parameters
task_complexity = 100_000   # Iterations per task (higher = more CPU intensive)
number_of_tasks = 10_000    # Total tasks to complete

print(f"Configuration:")
print(f"  Cores to use: {number_of_cores_to_use}")
print(f"  Task complexity: {task_complexity:,} iterations per task")
print(f"  Number of tasks: {number_of_tasks:,}")
print(f"  Total operations: {task_complexity * number_of_tasks:,}")

Configuration:
  Cores to use: 15
  Task complexity: 100,000 iterations per task
  Number of tasks: 10,000
  Total operations: 1,000,000,000


## Step 3: Define the CPU-Bound Task

In [5]:
def task(num):
    """
    Simulate a CPU-intensive task.

    The 'num' parameter isn't used in the computation — it exists because
    pool.map() requires a function that accepts one argument (it passes
    each value from the iterable). Every call performs the same work,
    which makes it easy to verify that serial and parallel produce
    identical results.

    In real applications, 'num' would typically be meaningful — e.g.,
    an index into a dataset, a file path, or a parameter for a simulation.
    """
    val = 0
    for i in range(task_complexity):
        val += i / 23
    return val

# Verify the function works
test_result = task(0)
print(f"Single task result: {test_result}")
print("Task function verified!")

Single task result: 217389130.43478262
Task function verified!


## Step 4: Sequential (Single-Process) Execution

Run all tasks on a single CPU core — the traditional Python approach:

In [6]:
%%time
print("Starting single-process (sequential) execution...")
print("-" * 50)

serial_start = time.time()

data_serial = []
for i in range(number_of_tasks):
    data_serial.append(task(i))

serial_time = time.time() - serial_start
print(f"Completed! Processed {len(data_serial):,} results")

Starting single-process (sequential) execution...
--------------------------------------------------
Completed! Processed 10,000 results
CPU times: user 22.5 s, sys: 2.35 ms, total: 22.5 s
Wall time: 22.5 s


> **Understanding the `%%time` output:**
>
> - **CPU times: user** — time spent executing your Python code on the CPU
> - **CPU times: sys** — time spent on system-level operations (memory allocation, I/O, etc.)
> - **CPU times: total** — user + sys combined
> - **Wall time** — the actual elapsed real-world time from start to finish (what you'd measure with a stopwatch)
>
> For CPU-bound tasks like ours, wall time and CPU total will be nearly identical. They diverge when your code spends time waiting (e.g., for network or disk I/O).

## Step 5: Parallel (Multi-Process) Execution

Now distribute the work across multiple CPU cores:

In [7]:
%%time
print(f"Starting multi-process execution with {number_of_cores_to_use} cores...")
print("-" * 50)

parallel_start = time.time()

# Create a pool of worker processes (like hiring a team of workers)
# The "with" statement ensures all processes are cleaned up when done
with Pool(number_of_cores_to_use) as pool:
    # pool.map() distributes the tasks across all worker processes:
    #   - Splits range(number_of_tasks) into chunks
    #   - Sends each chunk to a different process
    #   - Collects and combines the results in order
    data_parallel = pool.map(task, range(number_of_tasks))

parallel_time = time.time() - parallel_start
print(f"Completed! Processed {len(data_parallel):,} results")

# Verify results are identical
assert data_serial == data_parallel, "Results should be identical!"
print("Results verified: Serial and parallel produce identical outputs")

Starting multi-process execution with 15 cores...
--------------------------------------------------
Completed! Processed 10,000 results
Results verified: Serial and parallel produce identical outputs
CPU times: user 40.4 ms, sys: 81.1 ms, total: 121 ms
Wall time: 3.19 s


## Step 6: Analyze the Results

In [8]:
speedup = serial_time / parallel_time
efficiency = (speedup / number_of_cores_to_use) * 100

print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"  Serial time:   {serial_time:.2f} seconds")
print(f"  Parallel time: {parallel_time:.2f} seconds")
print(f"  Speedup:       {speedup:.1f}x")
print(f"  Cores used:    {number_of_cores_to_use}")
print(f"  Efficiency:    {efficiency:.0f}%")
print()
print("Why isn't speedup perfect?")
print("  - Process creation overhead")
print("  - Inter-process communication")
print("  - Memory copying between processes")
print("  - OS scheduling overhead")
print("  - Uneven work distribution")

PERFORMANCE COMPARISON
  Serial time:   22.48 seconds
  Parallel time: 3.19 seconds
  Speedup:       7.1x
  Cores used:    15
  Efficiency:    47%

Why isn't speedup perfect?
  - Process creation overhead
  - Inter-process communication
  - Memory copying between processes
  - OS scheduling overhead
  - Uneven work distribution


# From One Machine to Many: The Same Pattern at Scale

## What `Pool` Does on Your Machine

When you call `Pool(N)`, Python creates N independent worker processes. The `pool.map()` function splits your tasks into chunks, sends each chunk to a worker, and collects the results. Each worker has its own memory space — they don't share data directly.

```
┌─────────────────────────────────────────────────┐
│              Your Computer                      │
│                                                 │
│              ┌──────────────┐                   │
│              │ Pool Manager │                   │
│              │ (main process│                   │
│              │  splits work │                   │
│              │  & collects) │                   │
│              └──────┬───────┘                   │
│         ┌───────────┼───────────┐               │
│         ▼           ▼           ▼               │
│  ┌────────────┐ ┌────────────┐ ┌────────────┐  │
│  │ Process 1  │ │ Process 2  │ │ Process N  │  │
│  │ Own Memory │ │ Own Memory │ │ Own Memory │  │
│  │ Own GIL    │ │ Own GIL    │ │ Own GIL    │  │
│  └────────────┘ └────────────┘ └────────────┘  │
│      Core 1         Core 2         Core N       │
└─────────────────────────────────────────────────┘
```

## How Distributed Big Data Frameworks Work

In a distributed computing cluster, the pattern is the same — but instead of processes on one machine, the work is distributed across **many machines** connected over a network:

```
┌─────────────────────────────────────────────────┐
│              Distributed Cluster                │
│                                                 │
│              ┌──────────────┐                   │
│              │    Driver    │                   │
│              │ (coordinator │                   │
│              │  splits work │                   │
│              │  & collects) │                   │
│              └──────┬───────┘                   │
│          ┌──────────┼──────────┐                │
│          ▼          ▼          ▼                │
│  ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
│  │  Machine 1  │ │  Machine 2  │ │  Machine N  │
│  │  Own CPU    │ │  Own CPU    │ │  Own CPU    │
│  │  Own Memory │ │  Own Memory │ │  Own Memory │
│  │  Own Disk   │ │  Own Disk   │ │  Own Disk   │
│  └─────────────┘ └─────────────┘ └─────────────┘
│     Network ◄──────────────────────► Network    │
└─────────────────────────────────────────────────┘
```

## Side by Side

| | Multiprocessing (this lab) | Distributed Cluster |
|---|---|---|
| **Coordinator** | `Pool` manager process | Driver node |
| **Workers** | Processes on one machine | Processes on many machines |
| **Communication** | Inter-process (IPC) | Network (much slower) |
| **Data sharing** | Serialized via IPC | Serialized over network |
| **Scaling limit** | Number of CPU cores | Number of machines |
| **Failure impact** | Process crash = lost task | Node failure = lost tasks |

The key insight: `pool.map()` on your laptop and a distributed framework running on a 100-machine cluster are doing fundamentally the same thing — **splitting work across independent workers, each with their own memory, and combining the results**. The difference is that the cluster communicates over a network instead of IPC, which adds latency but removes the ceiling of a single machine's resources.

# Summary

In this lab you experienced the difference between sequential and parallel processing firsthand. Key takeaways:

1. **Python's GIL** prevents true parallelism with threads, but `multiprocessing` bypasses it by creating separate processes — each with its own interpreter and memory space.
2. **Parallel execution is significantly faster** for CPU-bound tasks, but speedup is never perfect due to overhead from process creation, inter-process communication, and OS scheduling.
3. **The pattern is the same at every scale.** Whether it's `pool.map()` distributing tasks across processes on your laptop, or a distributed framework distributing tasks across machines in a cluster, the approach is identical: a coordinator splits work, independent workers process it in their own memory space, and results are collected and combined.
4. **The only difference is the communication channel.** Locally, processes communicate via IPC. In a cluster, machines communicate over a network — which is slower, but allows you to scale beyond the limits of a single machine.

Try experimenting with `task_complexity` and `number_of_tasks` to see how the speedup changes!

# Going Further: Libraries That Make Parallelism Easier

In this lab we used `multiprocessing` directly to understand what happens under the hood. In practice, several Python libraries handle the complexity of splitting data, distributing work, and combining results for you:

- **[Dask](https://www.dask.org/)** — Scales pandas, NumPy, and scikit-learn workflows to multi-core or multi-machine. Handles chunking, task scheduling, and reductions automatically. A great stepping stone from single-machine Python to distributed computing.
- **[Ray](https://www.ray.io/)** — A general-purpose framework for parallelizing and distributing Python applications. Widely used for ML training, hyperparameter tuning, and serving models at scale.
- **[Joblib](https://joblib.readthedocs.io/)** — Lightweight library for simple parallel loops and caching. Used internally by scikit-learn (it's what powers the `n_jobs` parameter we saw earlier in this lab).
- **[concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html)** — Part of Python's standard library. Provides a cleaner API (`ThreadPoolExecutor` and `ProcessPoolExecutor`) on top of `threading` and `multiprocessing`.